# Building models with the `nn.Module`
*Intermediate Level*

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

--------------------------------

## Dropout layers
Dropout layers are a regularization technique used in neural networks to prevent overfitting.<br>
They work by randomly "dropping out" (setting to zero) a fraction of the neurons' outputs during training.<br>
This forces the network to learn more robust features that are not reliant on any single neuron.

In [3]:
# Example of dropout in a network
class TinyNet_WithDropout(nn.Module):
    def __init__(self, in_features=4, hidden=8, out_features=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), # 4 → 8
            nn.ReLU(),                      # Activation function
            nn.Dropout(0.5),                # 50% dropout
            nn.Linear(hidden, out_features) # 8 → 1
        )
    
    def forward(self, x):
        return self.net(x)

model_dropouts = TinyNet_WithDropout().to(device)               # Moves model to GPU if available, otherwise CPU
criterion = nn.MSELoss()                                        # Mean Squared Error: (prediction - target)²
optimizer = optim.Adam(model_dropouts.parameters(), lr=1e-2)

### How Dropout Works
**During Training (`model.train()`):**
1. Randomly sets neurons to zero with probability p (e.g., $\frac{1}{2}$ = 50%)
2. Scales remaining values by $\frac{1}{1-p}$ to maintain expected output
3. Different random pattern each forward pass

**During Evaluation (`model.eval()`)**:
1. No neurons are dropped - all neurons active
2. No scaling needed - uses learned weights directly

In [4]:
# Training mode
x = torch.randn(1, 5)
dropout = nn.Dropout(0.5)

print("Original:", x)
# tensor([[ 0.5, -1.2,  0.8, -0.3,  1.1]])

print("With dropout:", dropout(x))
# tensor([[ 0.0, -2.4,  0.0, -0.6,  2.2]])
#        ↑ zero  ↑ scaled  ↑ zero  ↑ scaled  ↑ scaled

Original: tensor([[-0.2187, -1.0749,  1.6030, -0.2454,  0.0124]])
With dropout: tensor([[-0.4374, -2.1497,  3.2059, -0.0000,  0.0249]])


**Why Dropout Prevents Overfitting**
1. Forces redundancy: Network can't rely on specific neurons
2. Reduces co-adaptation: Neurons learn more independent features
3. Ensemble effect: Each forward pass uses a different "sub-network"

In [32]:
model = TinyNet_WithDropout()

# Training mode - dropout active
model.train()
x = torch.randn(1, 4).to(device)
output1 = model(x)  # Some neurons randomly zeroed
output2 = model(x)  # Different neurons zeroed (randomness)

# Evaluation mode - dropout disabled  
model.eval()
output3 = model(x)  # All neurons active
output4 = model(x)  # Same as output3 (deterministic)

print(f"output1: {output1}")
print(f"output2: {output2}")
print(f"output3: {output3}")
print(f"output4: {output4}")

output1: tensor([[0.0559]], grad_fn=<AddmmBackward0>)
output2: tensor([[-0.1240]], grad_fn=<AddmmBackward0>)
output3: tensor([[0.0150]], grad_fn=<AddmmBackward0>)
output4: tensor([[0.0150]], grad_fn=<AddmmBackward0>)


Training with dropout leads to a model that generalizes better to unseen data.

In [6]:
# Same batches for all epochs
x_batch = torch.randn(32, 4).to(device)
y_batch = torch.randn(32, 1).to(device)

for epoch in range(500):
    optimizer.zero_grad()                 # (1) zero the gradients
    preds = model_dropouts(x_batch)       # (2) forward pass
    loss = criterion(preds, y_batch)      # (3) compute the loss, criterion = nn.MSELoss()
    loss.backward()                       # (4) Compute the gradients
    optimizer.step()                      # (5) Update parameters (weights and biases) based on gradients

    if epoch % 50 == 0:
        print(f"epoch {epoch:03d}  loss {loss.item():.4f}")


epoch 000  loss 1.1928
epoch 050  loss 0.7613
epoch 100  loss 0.6822
epoch 150  loss 0.6584
epoch 200  loss 0.6446
epoch 250  loss 0.6482
epoch 300  loss 0.6742
epoch 350  loss 0.4218
epoch 400  loss 0.4908
epoch 450  loss 0.4679


***Recap***:
- Dropout randomly deactivates neurons during training
- Helps prevent overfitting by promoting robust feature learning

In [7]:
# Training mode (default)
model.train()  # Enables dropout, batch norm training behavior

# Evaluation mode  
model.eval()   # Disables dropout, uses fixed batch norm stats

# Check current mode
print(model.training)  # True or False

False


--------------------------------

## Batch Processing
***All 32 Samples at Once***

When you have `x` with shape (32, 4), it means you have a batch of 32 samples, each with 4 features. <br>

During the forward pass, PyTorch processes **all 32 samples in parallel** through the network.<br>

For example, when you call `preds = model(x_batch)`, PyTorch applies the model to all 32 samples simultaneously,<br>
resulting in `preds` having shape (32, 1), which contains the predictions for each of the 32 samples.

In [53]:
# Input layer: nn.Linear(4, 8)
# Input: (32, 4) - 32 samples, 4 features each
# Output: (32, 8) - 32 samples, 8 hidden features each

# The math: 
# For each sample i: output[i] = input[i] @ weights + bias
# PyTorch does this for all 32 samples in parallel using matrix multiplication

### Loss Computation and Backpropagation with Batches

When computing the loss using a batch of data, such as with Mean Squared Error (MSE), the loss function calculates the error for each sample in the batch and then averages these errors to produce a single scalar loss value.

The gradient computation during backpropagation also considers all samples in the batch together,<br>
allowing the optimizer to update the model parameters **based on the average loss across the entire batch**.

#### Why Batch Processing?

1. **Efficiency**: GPU parallelization - processes multiple samples simultaneously
2. **Stable gradients**: Averages gradients across samples for more stable updates
3. **Memory efficient**: More efficient than processing one sample at a time

--------------------------------

## Export the model
**Model State and Checkpoints**

In [9]:
# Saving the weights and biases
torch.save(model_dropouts.state_dict(), 'model.pth')


# To load:
model = TinyNet_WithDropout()  # Create model instance first
model.load_state_dict(torch.load('model.pth'))
model.eval()  # Set to evaluation mode


TinyNet_WithDropout(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=8, out_features=1, bias=True)
  )
)

**Advantages**:
1. *Only saves parameters*: Weights and biases, not the class structure
2. *Portable*: Works across different scripts and environments
3. *Robust*: Less likely to break between PyTorch versions
4. *Smaller files*: Only stores the actual learned parameters

##### Checkpoint (metadata + state_dict)


In [11]:
# Save with additional metadata
checkpoint = {
    'model_state_dict': model_dropouts.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch,
    'loss': loss.item(),
    'model_config': {
        'in_features': 4,
        'hidden': 8,
        'out_features': 1
    }
}
torch.save(checkpoint, 'checkpoint.pth')

# Load with error handling
def load_checkpoint(filepath):
    try:
        checkpoint = torch.load(filepath)
        model = TinyNet_WithDropout().to(device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        return model, checkpoint
    except FileNotFoundError:
        print(f"Checkpoint {filepath} not found")
        return None, None


***Usage***

In [12]:
loaded_model , checkpoints = load_checkpoint('checkpoint.pth')
loaded_model

TinyNet_WithDropout(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [70]:
checkpoints['model_state_dict']

OrderedDict([('net.0.weight',
              tensor([[ 0.0711,  0.5151,  0.0613, -0.0643],
                      [-0.2228, -0.7912,  0.5742,  0.3461],
                      [-0.0163, -0.0814,  0.1013,  0.3737],
                      [-0.5937, -0.3272, -0.4675,  0.3088],
                      [-0.0326, -0.0266, -0.0984, -0.4732],
                      [ 0.2101, -0.2721, -0.1229, -0.1130],
                      [ 0.7389,  0.2996,  0.4285, -0.7566],
                      [-0.3755,  0.4810, -0.1369, -0.5464]])),
             ('net.0.bias',
              tensor([-1.2734, -0.9817, -1.0926, -1.0478, -0.8900, -0.9280, -1.1224, -0.8449])),
             ('net.3.weight',
              tensor([[-0.7610,  0.5346,  0.4770, -0.5635, -0.5096,  0.3022, -0.6893,  0.5089]])),
             ('net.3.bias', tensor([-0.1976]))])

In [72]:
checkpoint['optimizer_state_dict']

{'state': {0: {'step': tensor(1500.),
   'exp_avg': tensor([[ 1.3784e-23,  3.5817e-23,  2.3051e-24,  3.3773e-23],
           [ 2.1320e-02,  6.7900e-04,  7.6097e-03,  5.6770e-03],
           [-5.4942e-24,  2.6550e-23,  1.3524e-23,  4.2115e-23],
           [ 1.0038e-02,  2.9945e-03,  3.9255e-03, -1.9065e-03],
           [ 4.3700e-04,  3.3875e-03, -3.9623e-04,  5.4726e-03],
           [ 9.7753e-25, -3.2966e-24,  2.3435e-25, -2.8005e-24],
           [-1.9118e-03,  3.4175e-03, -2.0757e-03,  7.8770e-03],
           [-2.1964e-03, -1.0360e-03, -3.4066e-04, -1.7096e-03]]),
   'exp_avg_sq': tensor([[0.0024, 0.0050, 0.0044, 0.0027],
           [0.0019, 0.0017, 0.0011, 0.0023],
           [0.0061, 0.0098, 0.0076, 0.0210],
           [0.0013, 0.0023, 0.0016, 0.0033],
           [0.0009, 0.0009, 0.0005, 0.0023],
           [0.0003, 0.0005, 0.0003, 0.0006],
           [0.0012, 0.0008, 0.0010, 0.0016],
           [0.0021, 0.0021, 0.0023, 0.0024]])},
  1: {'step': tensor(1500.),
   'exp_avg': tensor([ 

--------------------------------

## Custom Forwards Pass
A custom forward pass gives you complete control over how data flows through your neural network.<br>
Instead of using `nn.Sequential`, you define the exact sequence of operations in the `forward()` method.


**Why Use Custom Forward Pass?**
1. **Complex architectures**: When you need skip connections, branching, or conditional logic
2. **Multiple inputs/outputs**: When your model takes multiple inputs or produces multiple outputs
3. **Dynamic behavior**: When the forward pass depends on input characteristics
4. **Debugging**: Easier to inspect intermediate values during development

In [4]:
# Sequential approach (simple but limited)
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(8, 1)
        )
    
    def forward(self, x):
        return self.net(x)

# Custom forward pass (more flexible)
class CustomNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.layer2 = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.layer1(x)      # Apply first linear layer
        x = self.relu(x)        # Apply activation
        x = self.dropout(x)     # Apply dropout
        x = self.layer2(x)      # Apply second linear layer
        return x

### Advanced Custom Forward Pass Example


##### 1. Skip connections (like in ResNets)

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.layer1 = nn.Linear(features, features)
        self.layer2 = nn.Linear(features, features)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        residual = x            # Save original input
        
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        
        x = x + residual        # Add skip connection
        x = self.relu(x)        # Final activation
        return x

##### 2. Multiple inputs/outputs

In [6]:
class MultiOutputNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Linear(4, 8)
        self.head1 = nn.Linear(8, 1)    # Regression head
        self.head2 = nn.Linear(8, 3)    # Classification head
        self.relu = nn.ReLU()
    
    def forward(self, x):
        shared_features = self.relu(self.shared(x))
        
        regression_output = self.head1(shared_features)
        classification_output = self.head2(shared_features)
        
        return regression_output, classification_output

##### 3. Conditional Logic

In [13]:
class AdaptiveNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.shallow_head = nn.Linear(8, 1)        # Simple path
        self.deep_layers = nn.Sequential(           # Complex path
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.layer1(x))      # Shared processing

        
        # Compute confidence measure
        confidence = torch.mean(torch.abs(x), dim=1)
        
        # Conditional logic: choose path based on confidence
        if self.training or confidence.mean() < 0.5:
            return self.deep_layers(x)             # Use complex path
        else:
            return self.shallow_head(x)            # Use simple path

The `AdaptiveNet` model demonstrates conditional logic in neural networks,<br>
where the forward pass makes decisions about which computational path to take<br>
based on the input characteristics or model state.

1. ***Shared Processing***
```python
x = self.relu(self.layer1(x)) # All inputs go through this first
```
- All inputs are processed through the first layer
- Creates common feature representation

2. ***Confidence Measure***
```python
confidence = torch.mean(torch.abs(x), dim=1)
```
- Computes a "confidence" score based on activation magnitudes
- Higher absolute values suggest the model is more "certain"
- `dim=1` averages across features for each sample in the batch

3. ***Confidence Measure***
```python
if self.training or confidence.mean() < 0.5:
    return self.deep_layers(x)      # Complex processing
else:
    return self.shallow_head(x)     # Simple processing
```
*Two conditions trigger the deep path*:
- **Training mode**: Always use complex path during training
- **Low confidence**: Use complex path when uncertain

#### Advantages of Conditional Logic
1. ***Computational Efficiency***
- Easy samples use fewer resources
- Hard samples get more computation
- Adaptive inference cost
2. ***Better Resource Allocation***
- Model capacity focused where needed
- Prevents over-processing simple inputs
- Dynamic complexity matching
3. ***Interpretability***
- Clear decision points in the model
- Understand why certain paths were taken
- Explainable AI benefits

#### Implementation Considerations

1. ***Batch Processing***
- Ensure all samples in a batch can be processed together
- Handle varying paths within the same batch


2. ***Training Challenges***
- Backpropagation through dynamic paths
- Ensure gradients flow correctly
- May require custom loss handling

In [ ]:
# Problem: Different paths during training vs inference
def forward(self, x):
    if self.training:
        # Always use full network during training
        return self.complex_path(x)
    else:
        # Use conditional logic during inference
        if confidence > threshold:
            return self.simple_path(x)
        else:
            return self.complex_path(x)

3. ***Gradient Flow***
- Ensure gradients are computed correctly through all paths
- Test backpropagation for all branches

In [14]:
# Ensure gradients flow to all components during training
def forward(self, x):
    shared = self.shared_layers(x)
    
    if self.training:
        # Train both paths
        simple_out = self.simple_path(shared)
        complex_out = self.complex_path(shared)
        
        # Use both outputs in loss calculation
        return simple_out, complex_out
    else:
        # Conditional inference
        if self.should_use_simple(shared):
            return self.simple_path(shared)
        else:
            return self.complex_path(shared)

4. ***Threshold Learning***
- Consider learning the confidence threshold
- Use a small network to predict threshold
- Allows model to adapt threshold during training

In [16]:
class LearnableThresholdNet(nn.Module):
    def __init__(self, threshold=0.5):
        super().__init__()
        self.threshold = torch.tensor(threshold)
        # ... other layers
    
    def forward(self, x):
        confidence = self.compute_confidence(x)
        
        if confidence.mean() > self.threshold:
            return self.simple_path(x)
        else:
            return self.complex_path(x)

#### Common Patterns
1. ***Binary Switching***
- Choose between two different paths
- Based on simple conditions
2. ***Multi-path Selection***
- Multiple expert networks
- Gating mechanisms decide routing
3. ***Early Termination***
- Stop computation when confident
- Reduce inference time
4. ***Adaptive Depth***
- Variable number of layers
- Based on input complexity
- Conditional logic makes neural networks more efficient and interpretable by allowing them to adapt their computation to the specific characteristics of each input!

--------------------------------

## Multiple Inputs

In [17]:
class MultiModalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder = nn.Linear(100, 64)
        self.image_encoder = nn.Linear(784, 64)
        self.fusion_layer = nn.Linear(128, 32)
        self.classifier = nn.Linear(32, 10)
        self.relu = nn.ReLU()
    
    def forward(self, text_input, image_input):
        text_features = self.relu(self.text_encoder(text_input))
        image_features = self.relu(self.image_encoder(image_input))
        
        # Concatenate features from different modalities
        fused_features = torch.cat([text_features, image_features], dim=1)
        
        x = self.relu(self.fusion_layer(fused_features))
        output = self.classifier(x)
        return output

--------------------------------

## Attention Mechanisms
Attention mechanisms allow neural networks to focus on specific parts of the input data when making predictions.<br>
They dynamically weigh the importance of different input elements, enabling the model to prioritize relevant information.

In [16]:
class AttentionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(4, 16)
        self.attention = nn.Linear(16, 1)
        self.decoder = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len, features)
        encoded = self.relu(self.encoder(x))  # (batch_size, seq_len, 16)
        
        # Compute attention weights
        attention_scores = self.attention(encoded)  # (batch_size, seq_len, 1)
        attention_weights = self.softmax(attention_scores.squeeze(-1))  # (batch_size, seq_len)
        
        # Apply attention
        attended = torch.sum(encoded * attention_weights.unsqueeze(-1), dim=1)  # (batch_size, 16)
        
        output = self.decoder(attended)
        return output

1. ***Feature Enchancement***
```python
encoded = self.relu(self.encoder(x))  
```
- Transform each position to richer representation
- Input:  (batch, 10, 4)  → Each timestep has 4 features
- Output: (batch, 10, 16) → Each timestep now has 16 enhanced features

2. ***Importance Scoring***
```python
attention_scores = self.attention(encoded)
```
- Generate scalar score for each position
- Input:  (batch, 10, 16) → 16 features per position
- Output: (batch, 10, 1)  → 1 importance score per position

3. ***Probability Normalization***
```python
attention_weights = self.softmax(attention_scores.squeeze(-1))
```
- Convert scores to probabilities (must sum to 1)
- Raw scores:`[2.1, -0.5, 1.8, 0.3, -1.2, ...]`
- After softmax:`[0.4,  0.05, 0.3, 0.1,  0.02, ...]` (sum = 1.0)

4. ***Weighted Combination***
```python
attended = torch.sum(encoded * attention_weights.unsqueeze(-1), dim=1)
```
- Create final representation by weighted average
- Higher weight positions contribute more to final result

In [ ]:
# Demonstrate with concrete numbers
def attention_walkthrough():
    model = AttentionNet()
    
    # Example: processing 3 sentences, each with 5 words, 4 features per word
    batch_text = torch.randn(3, 5, 4)
    
    with torch.no_grad():
        # Step 1: Encode each word
        encoded = model.relu(model.encoder(batch_text))
        print(f"Encoded features shape: {encoded.shape}")  # (3, 5, 16)
        
        # Step 2: Score each word's importance
        raw_scores = model.attention(encoded)
        print(f"Raw attention scores for first sentence: {raw_scores[0].squeeze()}")
        
        # Step 3: Normalize to probabilities
        weights = model.softmax(raw_scores.squeeze(-1))
        print(f"Attention weights for first sentence: {weights[0]}")
        print(f"Weights sum to: {weights[0].sum():.3f}")
        
        # Step 4: Create sentence representation
        sentence_repr = torch.sum(encoded * weights.unsqueeze(-1), dim=1)
        print(f"Final sentence representation shape: {sentence_repr.shape}")  # (3, 16)
        print("\t",sentence_repr[0])
        
        # The word with highest attention weight dominates the representation
        max_attention_word = torch.argmax(weights[0])
        print(f"Most important word position: {max_attention_word}")

attention_walkthrough()

Encoded features shape: torch.Size([3, 5, 16])
Raw attention scores for first sentence: tensor([0.3936, 0.0028, 0.7161, 0.1551, 0.0210])
Attention weights for first sentence: tensor([0.2206, 0.1492, 0.3045, 0.1738, 0.1520])
Weights sum to: 1.000
Final sentence representation shape: torch.Size([3, 16])
	 tensor([0.5126, 0.9114, 0.2697, 0.1229, 0.2085, 0.4641, 0.0125, 0.0000, 0.0611,
        0.0345, 0.0463, 0.0285, 0.0000, 0.0609, 0.6003, 0.1144])
Most important word position: 2


In [30]:
# Create model ONCE outside the function
model = AttentionNet()

def visualize_attention_consistent():
    # Use the SAME model instance and SAME data
    sentences = torch.randn(1, 8, 4)
    torch.manual_seed(42)  # Set seed for reproducible data
    
    with torch.no_grad():
        encoded = model.relu(model.encoder(sentences))
        scores = model.attention(encoded)
        weights = model.softmax(scores.squeeze(-1))
        
        print("Word importance in sentence (consistent):")
        word_names = ["The", "quick", "brown", "fox", "jumps", "over", "lazy", "dog"]
        
        for i, (word, weight) in enumerate(zip(word_names, weights[0])):
            bar = "█" * int(weight.item() * 50)
            print(f"{word:6s}: {weight.item():.3f} {bar}")

# Run multiple times - should be identical
visualize_attention_consistent()
print("\n" + "="*50 + "\n")

Word importance in sentence (consistent):
The   : 0.134 ██████
quick : 0.115 █████
brown : 0.117 █████
fox   : 0.105 █████
jumps : 0.133 ██████
over  : 0.109 █████
lazy  : 0.146 ███████
dog   : 0.141 ███████




### Key Concepts
1. Every attention mechanism computes weights
2. Weights determine how much each input contributes
3. Final output is weighted combination of inputs
```python
# Universal attention pattern:
scores = compute_attention_scores(query, keys)      # How relevant is each key?
weights = softmax(scores)                           # Normalize to probabilities  
output = weighted_sum(values, weights)   
```

## Gradient Clipping
Gradient clipping is a technique used to prevent exploding gradients during the training of neural networks.<br>
It involves capping the gradients to a maximum value, ensuring that they do not exceed a specified threshold.
``````python
# Example of gradient clipping in PyTorch
for epoch in range(100):
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds, y_batch)
    loss.backward()
    
    # Clip gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()

## Lerning Rate Scheduling
Learning rate scheduling is a technique used to adjust the learning rate during training.<br>
It helps improve convergence and can lead to better model performance by reducing the learning rate as training progresses.
```python
# Example of learning rate scheduling in PyTorch
from torch.optim.lr_scheduler import StepLR

optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

for epoch in range(100):
    # ... training code ...
    scheduler.step()  # Update learning rate

## Validation Loop
A validation loop is used to evaluate the performance of a model on a separate validation dataset during training.<br>
It helps monitor the model's generalization ability and detect overfitting.
```python
# Example of a validation loop in PyTorch
def validate_model(model, val_loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x_val, y_val in val_loader:
            preds = model(x_val)
            loss = criterion(preds, y_val)
            total_loss += loss.item()
    return total_loss / len(val_loader)

## Common Debugging Techniques
1. **Print Shapes**: Print tensor shapes at various points to ensure they match expected dimensions
2. **Check Gradients**: Verify that gradients are being computed and are not `None` or `NaN`
3. **Use Small Data**: Train on a small subset of data to quickly identify issues
4. **Visualize Outputs**: Plot intermediate outputs to inspect what the model is learning
5. **Unit Tests**: Write tests for individual components (layers, functions) to ensure correctness
6. **Check NaN in Loss**: Monitor loss values for NaN or infinite values